# Partial scheduling
If participant groups are large, the time taken to solve the optimisation problem is very long due to it scaling exponentially. E.g. it is estimate within [runtime.ipynb]('runtime.ipynb') that for 40 participants, it will lake ~ 200years to solve. 

As a workaround, instead of creating the full schedule, you can limit the forward plan to be fewer periods, e.g. the next 5 weeks. The history can then be used to constrain much of the optimisation problem for future weeks. 

In [1]:
import os
import sys
import time
import matplotlib.pyplot as plt
import numpy as np

module_path = os.path.abspath(os.path.join('../HouseBlend'))
if module_path not in sys.path:
    sys.path.append(module_path)
import HouseBlend as hb

## Setup the test case

In [29]:
parent_fullpath = os.path.join('../', 'example')  # the folder containing the input files (contacts.xlsx). Outputs will be saved here.

contacts, dates, availability = hb.import_contacts_df(folderpath=parent_fullpath, test=20)

schedule, bool_schedule = hb.import_schedules(folderpath=parent_fullpath, test=True)

Deleting previous contacts file
Creating new contact directory of 20 length for testing
Deleting previous schedule files
No existing schedule found. Ensure current_period is set to 1.


## Compare to if the first 3 weeks are set

In [13]:
n_periods = None
current_period = 18
start_time = time.time()
bool_schedule = hb.run_schedule_optimisation(contacts, bool_schedule, n_periods, availability)
end_time = time.time()
full_time = end_time - start_time
print("{:.1f}s".format(full_time))

24.1s


In [14]:
meeting_schedule = hb.generate_meeting_schedule(contacts, bool_schedule, folderpath=parent_fullpath, save=False)
meeting_schedule.head()

,Period 1,Period 2,Period 3,Period 4,Period 5,Period 6,Period 7,Period 8,Period 9,Period 10,Period 11,Period 12,Period 13,Period 14,Period 15,Period 16,Period 17,Period 18,Period 19
Person,,,,,,,,,,,,,,,,,,,
Jason Santos,Kelsey Anderson,Jonathan Frank,Danielle Stanley,Carolyn Evans,Nicole Cochran,Melanie Gillespie,Charlene Vazquez,Steven Duffy,Lindsey Ramirez,Kelsey Koch,Joel Williams,Angela Jackson,Angel Cox,Marc Cruz,Jennifer Smith,John Romero,Rebecca Oneill,Michael Cruz,Sheena Mendez
Lindsey Ramirez,Jennifer Smith,Nicole Cochran,Marc Cruz,John Romero,Michael Cruz,Angel Cox,Sheena Mendez,Jonathan Frank,Jason Santos,Melanie Gillespie,Angela Jackson,Charlene Vazquez,Kelsey Koch,Carolyn Evans,Danielle Stanley,Rebecca Oneill,Joel Williams,Kelsey Anderson,Steven Duffy
John Romero,Sheena Mendez,Marc Cruz,Rebecca Oneill,Lindsey Ramirez,Charlene Vazquez,Danielle Stanley,Jennifer Smith,Carolyn Evans,Steven Duffy,Joel Williams,Angel Cox,Kelsey Anderson,Melanie Gillespie,Jonathan Frank,Angela Jackson,Jason Santos,Kelsey Koch,Nicole Cochran,Michael Cruz
Sheena Mendez,John Romero,Carolyn Evans,Kelsey Koch,Melanie Gillespie,Jonathan Frank,Michael Cruz,Lindsey Ramirez,Charlene Vazquez,Joel Williams,Rebecca Oneill,Danielle Stanley,Jennifer Smith,Steven Duffy,Kelsey Anderson,Nicole Cochran,Marc Cruz,Angela Jackson,Angel Cox,Jason Santos
Jonathan Frank,Melanie Gillespie,Jason Santos,Joel Williams,Nicole Cochran,Sheena Mendez,Rebecca Oneill,Carolyn Evans,Lindsey Ramirez,Angel Cox,Marc Cruz,Michael Cruz,Danielle Stanley,Angela Jackson,John Romero,Kelsey Koch,Kelsey Anderson,Steven Duffy,Charlene Vazquez,Jennifer Smith


## Mimic real usage
1 week in advance, week by week

In [18]:
contacts, dates, availability = hb.import_contacts_df(folderpath=parent_fullpath, test=20)
schedule, bool_schedule = hb.import_schedules(folderpath=parent_fullpath, test=True)

Deleting previous contacts file
Creating new contact directory of 20 length for testing
Deleting previous schedule files
No existing schedule found. Ensure current_period is set to 1.


### Week 1

In [19]:
n_periods = 1
current_period = 1
start_time = time.time()
bool_schedule = hb.run_schedule_optimisation(contacts, bool_schedule, n_periods, availability)
end_time = time.time()
full_time = end_time - start_time
print("{:.1f}s".format(full_time))

meeting_schedule = hb.generate_meeting_schedule(contacts, bool_schedule, folderpath=parent_fullpath, save=True)
meeting_schedule.head()

1.0s


,Period 1
Person,
Jacqueline Mitchell,Lauren Moss
Alexis White,Kristen Jordan
Marie Gates,James Huynh
Valerie Green,Monica Mitchell
Steven Sanders,Jason Pacheco


### Week 2

In [20]:
n_periods = 1
current_week=2

# import previous schedule
schedule, bool_schedule = hb.import_schedules(folderpath=parent_fullpath)

start_time = time.time()
bool_schedule = hb.run_schedule_optimisation(contacts, bool_schedule, n_periods,
                                             availability,current_period=current_week)
end_time = time.time()
full_time = end_time - start_time
print("{:.1f}s".format(full_time))

meeting_schedule = hb.generate_meeting_schedule(contacts, bool_schedule, folderpath=parent_fullpath, save=True)
meeting_schedule.head()

Existing schedule exists, importing schedule
0.4s


,Period 1,Period 2
Person,,
Jacqueline Mitchell,Lauren Moss,Robin Perkins
Alexis White,Kristen Jordan,Austin Thompson
Marie Gates,James Huynh,Rebecca Gonzalez
Valerie Green,Monica Mitchell,Timothy Chavez
Steven Sanders,Jason Pacheco,Lauren Moss


### Week 3

In [21]:
n_periods = 1
current_week=3

# import previous schedule
schedule, bool_schedule = hb.import_schedules(folderpath=parent_fullpath)

start_time = time.time()
bool_schedule = hb.run_schedule_optimisation(contacts, bool_schedule, n_periods,
                                             availability,current_period=current_week)
end_time = time.time()
full_time = end_time - start_time
print("{:.1f}s".format(full_time))

meeting_schedule = hb.generate_meeting_schedule(contacts, bool_schedule, folderpath=parent_fullpath, save=True)
meeting_schedule.head()

Existing schedule exists, importing schedule
0.7s


,Period 1,Period 2,Period 3
Person,,,
Jacqueline Mitchell,Lauren Moss,Robin Perkins,Ernest Richards
Alexis White,Kristen Jordan,Austin Thompson,Jesse Avila
Marie Gates,James Huynh,Rebecca Gonzalez,Timothy Chavez
Valerie Green,Monica Mitchell,Timothy Chavez,Lauren Moss
Steven Sanders,Jason Pacheco,Lauren Moss,Robin Perkins


## Run the optimisation for 1 periods ahead, every week

In [22]:
meeting_schedules = []
s_time = time.time()
for period in range(4):
    n_periods = 1
    if period==0:
        schedule, bool_schedule = hb.import_schedules(folderpath=parent_fullpath, test=True)
    elif period>0:
        # import history
        schedule, bool_schedule = hb.import_schedules(folderpath=parent_fullpath)
    
    start_time = time.time()
    bool_schedule = hb.run_schedule_optimisation(contacts, bool_schedule, n_periods, availability, current_period=period+1)
    end_time = time.time()

    # save schedule
    meeting_schedule = hb.generate_meeting_schedule(contacts, bool_schedule, folderpath=parent_fullpath, save=True)
    print("Optimisation time: {:.1f}s".format(end_time-start_time))
    meeting_schedules.append(meeting_schedule)
overall_time = time.time() - s_time
print("Overall time {:.1f}s".format(overall_time))


Deleting previous schedule files
No existing schedule found. Ensure current_period is set to 1.
Optimisation time: 0.4s
Existing schedule exists, importing schedule
Optimisation time: 0.5s
Existing schedule exists, importing schedule
Optimisation time: 0.6s
Existing schedule exists, importing schedule
Optimisation time: 0.9s
Overall time 2.7s


In [28]:
meeting_schedules[3]

,Period 1,Period 2,Period 3,Period 4
Person,,,,
Jacqueline Mitchell,Lauren Moss,Robin Perkins,Ernest Richards,Valerie Green
Alexis White,Kristen Jordan,Austin Thompson,Jesse Avila,Erik Jacobs
Marie Gates,James Huynh,Rebecca Gonzalez,Timothy Chavez,Lauren Moss
Valerie Green,Monica Mitchell,Timothy Chavez,Lauren Moss,Jacqueline Mitchell
Steven Sanders,Jason Pacheco,Lauren Moss,Robin Perkins,Monica Mitchell
Rebecca Gonzalez,Timothy Chavez,Marie Gates,Ernest Kennedy,Christopher Humphrey
Timothy Chavez,Rebecca Gonzalez,Valerie Green,Marie Gates,Robin Perkins
Monica Mitchell,Valerie Green,Jason Pacheco,Austin Thompson,Steven Sanders
Randall Owens,Jesse Avila,James Huynh,Kristen Jordan,Austin Thompson


## Run the optimisation for 4 periods (e.g. weeks) ahead as a baseline

In [30]:
schedule, bool_schedule = hb.import_schedules(folderpath=parent_fullpath, test=True)
n_periods = 4
current_period = 1
start_time = time.time()
bool_schedule = hb.run_schedule_optimisation(contacts, bool_schedule, n_periods, availability,
                                            multiple_meetings='strict')
end_time = time.time()
full_time = end_time - start_time
print("{:.1f}s".format(full_time))

Deleting previous schedule files
No existing schedule found. Ensure current_period is set to 1.
1.0s


In [31]:
meeting_schedule = hb.generate_meeting_schedule(contacts, bool_schedule, folderpath=parent_fullpath, save=False)
meeting_schedule.head(3)

,Period 1,Period 2,Period 3,Period 4
Person,,,,
Heather Jones,Kendra Lane,Meredith Nelson,Steven Sparks,Jessica Rios
Jeffrey Wilkinson,Tammie Barron,Brittany Powell,Frank Brown,Meredith Nelson
Debbie Hill,Cheryl Gonzalez,Justin Brown,Jessica Rios,Brittany Powell
